In [1]:
from bs4 import BeautifulSoup as bs
import requests as rq
import pandas as pd
import time
from sqlalchemy import create_engine, text

In [2]:
url = "https://alinino.az/collection/knigi-na-angliyskom-yazyke"

In [3]:
response = rq.get(url)
response

<Response [200]>

In [4]:
soup = bs(response.text, "html.parser")

In [5]:
prices = soup.find_all("span", class_="product-card__price")
prices_old = soup.find_all("span", class_="product-card__old-price")
names = soup.find_all("a", class_="product-card__title")


In [6]:
prices[1].text.strip().split()[0] # price
prices_old[0].text.strip().split()[0] # old price
prices[1].text.strip().split()[1] # currency
names[0].text.strip() # name

"S for Sufi A Beginner's Guide to Sufi Spirituality"

In [7]:
names_list = []
prices_list = []
old_prices_list = []
currency_list = []


for i in range(len(names)):
    names_list.append(names[i].text.strip())
    prices_list.append(prices[i].text.strip().split()[0])
    old_prices_list.append(prices_old[i].text.strip().split()[0])
    currency_list.append(prices[i].text.strip().split()[1])

In [8]:
data = {
    'Product Name': names_list,
    'Current Price': prices_list,
    'Old Price': old_prices_list,
    'Currency': currency_list
}

In [9]:
df = pd.DataFrame(data)
df.index = pd.RangeIndex(start=1, stop=len(df) + 1)

In [10]:
df.head()

,Product Name,Current Price,Old Price,Currency
1,S for Sufi A Beginner's Guide to Sufi Spiritua...,30.12,31.70,AZN
2,A Practical Guide to Project Finance,29.36,30.90,AZN
3,The Symposium,9.12,9.60,AZN
4,Really Big Questions For Daring Thinkers: Over...,19.19,20.20,AZN
5,Science Of Spice,52.06,54.80,AZN


In [11]:
def scrape1(length=1):

    url = "https://alinino.az/collection/knigi-na-angliyskom-yazyke"
    response = rq.get(url)
    soup = bs(response.text, "html.parser")

    prices = soup.find_all("span", class_="product-card__price")
    prices_old = soup.find_all("span", class_="product-card__old-price")
    names = soup.find_all("a", class_="product-card__title")

    prices[1].text.strip().split()[0] # price
    prices_old[0].text.strip().split()[0] # old price
    prices[1].text.strip().split()[1] # currency
    names[0].text.strip() # name

    names_list = []
    prices_list = []
    old_prices_list = []
    currency_list = []


    for i in range(len(names)):
        names_list.append(names[i].text.strip())
        prices_list.append(prices[i].text.strip().split()[0])
        old_prices_list.append(prices_old[i].text.strip().split()[0])
        currency_list.append(prices[i].text.strip().split()[1])

    data = {
    'Product Name': names_list,
    'Current Price': prices_list,
    'Old Price': old_prices_list,
    'Currency': currency_list
    }

    df = pd.DataFrame(data)
    df.index = pd.RangeIndex(start=1, stop=len(df) + 1)

    return df
    

In [12]:
df = scrape1()

In [13]:
df.head()

,Product Name,Current Price,Old Price,Currency
1,S for Sufi A Beginner's Guide to Sufi Spiritua...,30.12,31.70,AZN
2,A Practical Guide to Project Finance,29.36,30.90,AZN
3,The Symposium,9.12,9.60,AZN
4,Really Big Questions For Daring Thinkers: Over...,19.19,20.20,AZN
5,Science Of Spice,52.06,54.80,AZN


In [22]:
def scrape_page(page: int):
    if page == 1:
        url = "https://alinino.az/collection/knigi-na-angliyskom-yazyke"
    else:
        url = f"https://alinino.az/collection/knigi-na-angliyskom-yazyke?page={page}"

    response = rq.get(url, timeout=10)
    response.raise_for_status()

    soup = bs(response.text, "html.parser")

    names = soup.find_all("a", class_="product-card__title")
    prices = soup.find_all("span", class_="product-card__price")
    # old_prices = soup.find_all("span", class_="product-card__old-price")

    data = []

    for i in range(len(names)):
        name = names[i].text.strip()

        price_parts = prices[i].text.strip().split()
        current_price = price_parts[0]
        currency = price_parts[1] if len(price_parts) > 1 else None

        # # Old price may be missing
        # old_price = (
        #     old_prices[i].text.strip().split()[0]
        #     if i < len(old_prices)
        #     else None
        # )

        data.append({
            "Product Name": name,
            "Current Price": current_price,
            #"Old Price": old_price,
            "Currency": currency
        })

    return pd.DataFrame(data)


def scrape_all_pages(start=1, end=412):
    all_data = []

    for page in range(start, end + 1):
        print(f"Scraping page {page}...")
        df = scrape_page(page)
        all_data.append(df)

        time.sleep(3)  # break for a second

    final_df = pd.concat(all_data, ignore_index=True)
    final_df.index += 1
    return final_df

In [15]:
# df = scrape_all_pages(1, 412)

In [24]:
def upload_to_sql(
    df: pd.DataFrame,
    server: str,
    database: str,
    table_name: str,
    username: str,
    password: str,
    driver: str = "ODBC Driver 17 for SQL Server"
):
    connection_string = (
        f"mssql+pyodbc://{username}:{password}@{server}/{database}"
        f"?driver={driver.replace(' ', '+')}"
    )

    engine = create_engine(connection_string, fast_executemany=True)

    with engine.begin() as conn:
        # Drop table if exists
        conn.execute(text(f"""
            IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
                DROP TABLE {table_name};
        """))

        # Create & insert
        df.to_sql(
            table_name,
            con=conn,
            index=False,
            if_exists="replace"  # safe after drop
        )

    print(f"Table '{table_name}' uploaded successfully.")


In [ ]:
def upload_to_postgres(
    df: pd.DataFrame,
    host: str,
    database: str,
    table_name: str,
    username: str,
    password: str,
    port: int = 5432
):
    connection_string = (
        f"postgresql+psycopg2://{username}:{password}"
        f"@{host}:{port}/{database}"
    )

    engine = create_engine(connection_string)

    with engine.begin() as conn:
        # Drop table if exists
        conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))

        # Create & insert
        df.to_sql(
            table_name,
            con=conn,
            index=False,
            if_exists="replace"
        )

    print(f"Table '{table_name}' uploaded successfully.")


In [ ]:
df = scrape_all_pages(1, 50)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...


In [ ]:
upload_to_postgres(
    df=df,
    host="localhost",
    port=5434,
    database="books_db",
    table_name="alinino_books",
    username="books_user",
    password="books_password"
)

Table 'alinino_books' uploaded successfully.
